## Karthik Aggarwal - Final Project

**Pitch:**

In September 2026, the South Bronx was the latest site in a string of recent Legionnaires Disease outbreaks within New York City. In the Bronx outbreak, 24 New Yorkers tested positive for Legionnaires and two tragically passed. Legionnaires outbreaks within New York City appear ever-increasing in recent years: In July 2026, an Upper East Side outbreak resulted in 11 dead and in July-August 2025 a Central Harlem outbreak caused the deaths of seven New Yorkers.

Using cooling tower inspection data from NYC Open Records, I am pitching a data investigation into whether the Bronx has kept pace with Manhattan in complying with the Dept. of Health's inspection regulations for Legionnaires Disease prevention. I am examining five years' worth of City inspection data, from 2022-2026, across the Bronx and Manhattan to hypothesize there is an ongoing divide in compliance between the boroughs, likely due to socioeconomic factors.

Indeed, an early analysis of data from 2022-2026 shows that 24.2 percent of Bronx inspections found at least one inspection violation, compared with 17.6 percent in Manhattan. The disparity is sharper for the most serious category: 12.0 percent of Bronx inspections cited a Public Health Hazard, versus 7.7 percent in Manhattan, meaning a Bronx inspection was about 56 percent more likely to find one. I will develop the Open Records data out further using graphs, tables, and charts to show how inspection results data suggests Bronx residents face a greater public health risk.

Note: Open Records data for 2026 inspections is complete through Nov. 17, 2026.

**Sources:** 

https://data.cityofnewyork.us/Health/NYC-Cooling-Tower-System-Inspection-Results/f9wb-g8mb/about_data

https://www.nyc.gov/site/doh/health/health-topics/legionnaires-disease.page

https://www.cbsnews.com/newyork/news/upper-east-side-legionnaires-disease-death-toll-august-2026/

https://evidence.nejm.org/doi/full/10.1056/EVIDpha2600062?referrer=https%3A%2F%2Fwww.google.com%2F


In [2]:
import glob
import pandas as pd

CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
print("Using:", CSV)

Using: NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv


In [7]:
import glob
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display

# ---------- STEP 0: load ----------
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
print("Using:", CSV, "\n")

df = pd.read_csv(CSV)
df["date"] = pd.to_datetime(df["Inspection_date"])
df["year"] = df["date"].dt.year
df["has_violation"] = df["Violation_Code"].notna()   # a violation = a row with a violation code

# ---------- FIGURE 1: dataset size ----------
total_rows = len(df)
violation_entries = df["has_violation"].sum()
print("FIGURE 1: dataset size")
print(f"  Total rows in file:            {total_rows:,}")
print(f"  Rows with a violation:         {violation_entries:,}")
print(f"  Rows with no violation cited:  {total_rows - violation_entries:,}")
print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# ---------- FIGURE 2: collapse to one row per inspection ----------
inspections = (
    df.groupby(["System_ID", "Inspection_date", "Borough", "year", "Inspection_Type"])
    .agg(
        any_violation=("has_violation", "max"),
        phh=("Violation_Type", lambda s: (s == "PHH").any()),
    )
    .reset_index()
)
print("\nFIGURE 2: inspections")
print(f"  Unique inspections: {len(inspections):,}")

# ---------- FIGURES 3-4: rates by borough ----------
START, END = 2022, 2026          # change to (2022, 2025) to exclude 2026
window = inspections[inspections["year"].between(START, END)]

table = window.groupby("Borough").agg(
    inspections=("any_violation", "size"),
    with_violation=("any_violation", "sum"),
    with_phh=("phh", "sum"),
)
table["pct_any_violation"] = (table["with_violation"] / table["inspections"] * 100).round(1)
table["pct_phh"] = (table["with_phh"] / table["inspections"] * 100).round(1)

table = table.rename(columns={
    "inspections": "Inspections",
    "with_violation": "With a Violation",
    "with_phh": "With a Public Health Hazard",
    "pct_any_violation": "% with a Violation",
    "pct_phh": "% with a Public Health Hazard",
})
table.index.name = "Borough"

print(f"\nFIGURES 3-4: {START}-{END} rates")
display(table.style.format({
    "Inspections": "{:,}",
    "With a Violation": "{:,}",
    "With a Public Health Hazard": "{:,}",
    "% with a Violation": "{:.1f}",
    "% with a Public Health Hazard": "{:.1f}",
}))

# ---------- Significance tests ----------
print("Significance tests (chi-square):")
for col, label in [("any_violation", "any violation"), ("phh", "public health hazard")]:
    p = chi2_contingency(pd.crosstab(window["Borough"], window[col]))[1]
    print(f"  {label}: p = {p:.2g}")

Using: NYC_Cooling_Tower_System_Inspection_Results_20261122 Manhattan and Bronx.csv 

FIGURE 1: dataset size
  Total rows in file:            95,893
  Rows with a violation:         67,975
  Rows with no violation cited:  27,918
  Date range: 2017-01-17 to 2026-10-16

FIGURE 2: inspections
  Unique inspections: 29,450

FIGURES 3-4: 2022-2026 rates


,Inspections,With a Violation,With a Public Health Hazard,% with a Violation,% with a Public Health Hazard
Borough,,,,,
Bronx,"1,064",257,128,24.2,12.0
Manhattan,"11,193","1,966",862,17.6,7.7


Significance tests (chi-square):
  any violation: p = 1.2e-07
  public health hazard: p = 9.9e-07


_The difference between the boroughs is statistically significant; it's very unlikely to be due to chance._

The table above shows that in the last five years a greater percentage of inspections in the Bronx (24.2%) have resulted in an inspection violation than in Manhattan, even though there were 165% more inspections in Manhattan than the Bronx. 

Crucially, this difference also extends to the percentage of violations in the Bronx (12%) deemed a Public Health Hazard--the DOH's highest violation. 

Now I'm going to also look at violations for both boroughs that were deemed Critical, one level less serious than Public Health Hazard, and General, the least serious infraction, and the percentage difference in each.

In [9]:
import glob
import pandas as pd
from IPython.display import display

# ---------- Load ----------
CSV = glob.glob("**/*Manhattan*Bronx*.csv", recursive=True)[0]
df = pd.read_csv(CSV)
df["year"] = pd.to_datetime(df["Inspection_date"]).dt.year

START, END = 2022, 2026
window = df[df["year"].between(START, END)]

# ---------- Violations by type and borough ----------
viol = window[window["Violation_Type"].notna()]
counts = pd.crosstab(viol["Borough"], viol["Violation_Type"])[["General", "Critical", "PHH"]]
counts.columns = ["General", "Critical", "Public health hazard"]
counts["All types"] = counts.sum(axis=1)

# Unique inspections per borough
n_insp = (
    window.drop_duplicates(["System_ID", "Inspection_date", "Inspection_Type"])
    .groupby("Borough").size()
)
per100 = counts.div(n_insp, axis=0) * 100

# ---------- Comparison table ----------
comparison = pd.DataFrame({
    "Bronx violations": counts.loc["Bronx"],
    "Manhattan violations": counts.loc["Manhattan"],
    "Bronx per 100 inspections": per100.loc["Bronx"],
    "Manhattan per 100 inspections": per100.loc["Manhattan"],
})
comparison["% difference (Bronx vs. Manhattan)"] = (
    (comparison["Bronx per 100 inspections"] - comparison["Manhattan per 100 inspections"])
    / comparison["Manhattan per 100 inspections"] * 100
)
comparison.index.name = "Violation type"

print(f"Violations issued, {START}-{END}")
display(comparison.style.format({
    "Bronx violations": "{:,.0f}",
    "Manhattan violations": "{:,.0f}",
    "Bronx per 100 inspections": "{:.1f}",
    "Manhattan per 100 inspections": "{:.1f}",
    "% difference (Bronx vs. Manhattan)": "{:+.1f}%",
}))


Violations issued, 2022-2026


,Bronx violations,Manhattan violations,Bronx per 100 inspections,Manhattan per 100 inspections,% difference (Bronx vs. Manhattan)
Violation type,,,,,
General,288,"1,414",27.1,12.6,+114.3%
Critical,576,"3,313",54.1,29.6,+82.9%
Public health hazard,160,"1,042",15.0,9.3,+61.5%
All types,"1,024","5,769",96.2,51.5,+86.7%


For every 100 inspections, Bronx cooling towers were cited for about 96 violations, compared with about 52 in Manhattan. The overall percentage difference between the boroughs reached a whopping 87% among all types of violations, with Bronx cooling towers being in greater violation of DOH regulations.